# 🚗 SmartEV - EV Driving Range Prediction Model Training Pipeline
### Research Component: Intelligent EV Charging Optimization & Range Prediction
**Student ID**: IT22134080  
**Platform**: Google Colab / Local Jupyter Notebook  

This notebook contains the complete 16-step machine learning development pipeline:
1. Library Imports
2. Dataset Loading
3. Dataset Inspection
4. Data Cleaning & Type Casting
5. Missing Value Handling
6. Outlier Detection & Handling
7. Exploratory Data Analysis (EDA)
8. Feature Engineering (Environmental & Speed Factors)
9. Feature Selection
10. Train/Test Split
11. Baseline Model Training (Linear Regression / Ridge)
12. Advanced Model Training (Random Forest / Gradient Boosting / XGBoost)
13. Evaluation Metrics (MAE, RMSE, R²)
14. Model Comparison & Visualization
15. Best Champion Model Selection
16. Model Export (`joblib`) for Drop-in Inference

## Step 1: Import Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

%matplotlib inline
print("Libraries imported successfully.")

## Step 2: Load Dataset

In [ ]:
# Load dataset from local path or upload to Colab
csv_path = "../datasets/raw/ev_range_dataset_sample.csv"
if not os.path.exists(csv_path):
    csv_path = "ev_range_dataset_sample.csv"

df = pd.read_csv(csv_path)
print(f"Dataset shape: {df.shape}")
df.head()

## Step 3 & 4: Inspect & Clean Dataset

In [ ]:
print("Data summary:")
print(df.info())
print("\nSummary statistics:")
df.describe()

## Step 5: Missing Value Handling

In [ ]:
missing_counts = df.isnull().sum()
print("Missing values per column:\n", missing_counts)
# Impute or drop if any missing
df.fillna(df.median(numeric_only=True), inplace=True)

## Step 6: Outlier Analysis

In [ ]:
# Verify values within physical boundaries
df = df[(df['soc'] >= 0) & (df['soc'] <= 100)]
df = df[df['speed_kmh'] >= 0]
df = df[df['battery_capacity_kwh'] > 0]
print(f"Cleaned dataset shape: {df.shape}")

## Step 7: Exploratory Data Analysis (EDA)

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x='soc', y='remaining_range_km', hue='speed_kmh', palette='viridis')
plt.title('State of Charge (SoC) vs. Remaining Range (km)')
plt.xlabel('Battery SoC (%)')
plt.ylabel('Remaining Range (km)')
plt.show()

## Step 8 & 9: Feature Engineering & Selection

In [ ]:
# Usable energy interaction feature
df['usable_energy_kwh'] = df['battery_capacity_kwh'] * (df['soc'] / 100.0)

feature_cols = [
    'soc',
    'battery_capacity_kwh',
    'speed_kmh',
    'temperature_c',
    'energy_consumption_kwh_per_100km'
]
target_col = 'remaining_range_km'

X = df[feature_cols]
y = df[target_col]

## Step 10: Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")

## Step 11 & 12: Baseline & Advanced Model Training

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    results[name] = {"Model": model, "MAE": mae, "RMSE": rmse, "R2": r2}
    print(f"{name:20s} -> MAE: {mae:.2f} km | RMSE: {rmse:.2f} km | R2: {r2:.4f}")

## Step 13 & 14: Model Comparison & Evaluation

In [ ]:
eval_df = pd.DataFrame([
    {"Model": k, "MAE (km)": round(v["MAE"], 2), "RMSE (km)": round(v["RMSE"], 2), "R2": round(v["R2"], 4)}
    for k, v in results.items()
])
eval_df.sort_values(by="R2", ascending=False, inplace=True)
print("Model Evaluation Benchmark Table:")
eval_df

## Step 15 & 16: Champion Model Selection & Drop-in Export

In [ ]:
best_model_name = eval_df.iloc[0]["Model"]
champion_model = results[best_model_name]["Model"]
print(f"Selected Champion Model: {best_model_name}")

export_path = "ev_range_model.joblib"
joblib.dump(champion_model, export_path)
print(f"Exported model to {export_path}.")
print("\n--- Drop-in Instructions ---")
print("1. Download 'ev_range_model.joblib'")
print("2. Copy into 'ml-service/saved_models/range/' folder in your project.")
print("3. The ML microservice will automatically detect and load this model with zero code changes!")